# DE2 - Final Project : NYC Yellow Taxi Pipeline

**Track B:** NYC Yellow Taxi Trip Records - January 2026 
**Platform:** PySpark 4.0.1 - Single machine  
**Authors:** Rabahi Enzo - Couzinet Lorenzo

---

## Table of Contents

- [1 - Configuration & Spark Session](#1---configuration--spark-session) - 2 cells
- [2 - Bronze Layer – Raw Immutable Landing](#2-bronze-layer--raw-immutable-landing) - 5 cells
- [3 - Silver Layer – Cleaning, Typing, Schema Contracts, Deduplication](#3-silver-layer--cleaning-typing-schema-contracts-deduplication) - 8 cells
- [4 - Gold Layer – Analytical Tables](#4-gold-layer--analytical-tables) - 9 cells
- [5 - Streaming Ingestion – Structured Streaming with Windowed Aggregation](#5-streaming-ingestion--structured-streaming-with-windowed-aggregation) - 7 cells
- [6 - Text Processing – Tokenisation, TF-IDF & Inverted Index](#6-text-processing--tokenisation-tf-idf--inverted-index) - 10 cells
- [7 - Iterative Workload – KMeans & BisectingKMeans Clustering](#7-iterative-workload--kmeans--bisectingkmeans-clustering) - 10 cells
- [8 - LLM Data Preparation – Curated Dataset for Fine-tuning / RAG](#8-llm-data-preparation--curated-dataset-for-fine-tuning--rag) - 20 cells


---

## **1 - Configuration & Spark Session**

In [ ]:
#  Imports 
import time, csv, datetime, io, sys, json, shutil, warnings
from pathlib import Path

import yaml
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, when, isnull,
    min as fmin, max as fmax,
    avg, hour, sum as fsum,
    round as spark_round, window,
    upper, trim,
    year as spark_year,
    row_number,
    to_date, date_format,
    unix_timestamp as uts,
    lit, concat_ws, cast,
    monotonically_increasing_id,
    explode, collect_list, size,
    format_string, length,
    sha2, to_json, struct,
    lower as flower,
    udf,
)
from pyspark.sql.types import IntegerType, DoubleType, BooleanType
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer, StopWordsRemover,
    HashingTF, IDF,
    VectorAssembler, StandardScaler,
)
from pyspark.ml.clustering import KMeans, BisectingKMeans
from pyspark.ml.evaluation import ClusteringEvaluator

warnings.filterwarnings("ignore")

# Load config 
CONFIG_PATH = Path("de2_project_config.yml")
with open(CONFIG_PATH, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

#  Key paths 
RAW_FILE     = Path(cfg["data"]["raw_file"])
BRONZE_DIR   = Path(cfg["paths"]["raw_data"])
SILVER_DIR   = Path(cfg["paths"]["silver"])
GOLD_DIR     = Path(cfg["paths"]["gold"])
PROOF_DIR    = Path(cfg["paths"]["proof"])
METRICS_FILE = Path(cfg["metrics"])

for d in [BRONZE_DIR, SILVER_DIR, GOLD_DIR, PROOF_DIR, METRICS_FILE.parent]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Raw file     : {RAW_FILE}")
print(f"Bronze dir   : {BRONZE_DIR.resolve()}")
print(f"Silver dir   : {SILVER_DIR.resolve()}")
print(f"Config loaded")


Raw file     : yellow_tripdata_2026-01.parquet
Bronze dir   : C:\Users\Enzor\Downloads\outputs\project\bronze
Silver dir   : C:\Users\Enzor\Downloads\outputs\project\silver
Config loaded


In [2]:
spark = (
    SparkSession.builder
    .appName(cfg["spark"]["app_name"])
    .master(cfg["spark"]["master"])
    .config("spark.driver.memory",          cfg["spark"]["driver_memory"])
    .config("spark.sql.shuffle.partitions", cfg["spark"]["sql_shuffle_partitions"])
    .config("spark.sql.parquet.enableVectorizedReader", "true")
    .config("spark.hadoop.fs.permissions", "false")
    .config("spark.hadoop.fs.permissions.enabled", "false")
    .config("spark.hadoop.hadoop.home.dir", str(HADOOP_HOME))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version :", spark.version)
print("Spark UI      :", spark.sparkContext.uiWebUrl)

Spark version : 4.1.2
Spark UI      : http://DESKTOP-M9BCDL0.home:4040


---
## 2. Bronze Layer - Raw Immutable Landing <a id='2'></a>

**Principle:** the raw file is copied as-is into `bronze/` and never modified.  
All transformations begin from Silver. We validate schema, null rates and row count here.


In [3]:
#  Copy raw file into bronze/ (immutable landing)   
bronze_dest = BRONZE_DIR / RAW_FILE.name
if not bronze_dest.exists():
    shutil.copy2(RAW_FILE, bronze_dest)
    print(f"[COPY] {RAW_FILE.name} -> {bronze_dest}")
else:
    print(f"[SKIP] {bronze_dest.name} already in bronze/")

size_mb = bronze_dest.stat().st_size / 1e6
print(f"File size    : {size_mb:.1f} MB")

[SKIP] yellow_tripdata_2026-01.parquet already in bronze/
File size    : 64.2 MB


In [4]:
#  Read Bronze with Spark 
t0 = time.time()
df_bronze = spark.read.parquet(str(bronze_dest))
bronze_rows = df_bronze.count()
print(f"Bronze rows  : {bronze_rows:,}  ({time.time()-t0:.1f}s)")
print(f"Partitions   : {df_bronze.rdd.getNumPartitions()}")
print()
print("Schema")
df_bronze.printSchema()

Bronze rows  : 3,724,889  (3.1s)
Partitions   : 12

Schema
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
#  Null counts 
print(" Null counts per column ")
df_bronze.select([
    count(when(isnull(c), c)).alias(c) for c in df_bronze.columns
]).show(truncate=False)

#  Date range 
print(" Date range ")
df_bronze.select(
    fmin("tpep_pickup_datetime").alias("min_pickup"),
    fmax("tpep_pickup_datetime").alias("max_pickup")
).show()

#  Key distributions 
print(" Key distributions ")
df_bronze.select(
    avg("fare_amount").alias("avg_fare"),
    avg("trip_distance").alias("avg_dist"),
    avg("total_amount").alias("avg_total"),
).show()

 Null counts per column 
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|0       |0                   |0                    |1088058        |0            |1088058   |1088058    

In [6]:
#  Bronze footprint & SLO check 
bronze_size_gb = bronze_dest.stat().st_size / 1e9

print(f"Bronze size  : {bronze_size_gb:.4f} GB  ({size_mb:.1f} MB)")
print(f"Total rows   : {bronze_rows:,}")

slo_rows = bronze_rows >= cfg["data"]["min_rows"]
print(f"\nSLO ≥ {cfg['data']['min_rows']:,} rows -> {' MET' if slo_rows else 'Warning  NOT MET (single month - acceptable)'} ({bronze_rows:,})")

Bronze size  : 0.0642 GB  (64.2 MB)
Total rows   : 3,724,889

SLO ≥ 1,000,000 rows ->  MET (3,724,889)


In [7]:
#  Record Bronze metrics 
metrics_bronze = {
    "timestamp"  : datetime.datetime.now().isoformat(),
    "stage"      : "bronze_ingestion",
    "rows"       : bronze_rows,
    "size_gb"    : round(bronze_size_gb, 4),
    "partitions" : df_bronze.rdd.getNumPartitions(),
    "note"       : f"Raw immutable landing - {RAW_FILE.name}"
}
write_header = not METRICS_FILE.exists()
with open(METRICS_FILE, "a", newline="") as mf:
    w = csv.DictWriter(mf, fieldnames=metrics_bronze.keys())
    if write_header:
        w.writeheader()
    w.writerow(metrics_bronze)

print("Bronze metrics written ->", METRICS_FILE)
for k, v in metrics_bronze.items():
    print(f"  {k:12s}: {v}")

Bronze metrics written -> outputs\project\metrics\project_metrics_log.csv
  timestamp   : 2026-06-07T20:16:43.494238
  stage       : bronze_ingestion
  rows        : 3724889
  size_gb     : 0.0642
  partitions  : 12
  note        : Raw immutable landing - yellow_tripdata_2026-01.parquet


---
## 3. Silver Layer - Cleaning, Typing, Schema Contracts, Deduplication <a id='3'></a>

### Pipeline steps
| Step | Description |
|------|-------------|
| 3.1 | Cast & enforce schema contracts (types, domains) |
| 3.2 | Filter invalid records (business rules from NYC TLC data dictionary) |
| 3.3 | Deduplicate on natural key `(VendorID, tpep_pickup_datetime, PULocationID)` |
| 3.4 | Add derived columns (`pickup_date`, `pickup_month`, `trip_duration_min`, `avg_speed_mph`) |
| 3.5 | Capture `EXPLAIN` physical plan (proof) |
| 3.6 | Write partitioned Silver |
| 3.7 | Footprint comparison & metrics |


### 3.1 - Schema Contracts & Casting

In [8]:
df_cast = (
    df_bronze
    # Integer fields
    .withColumn("VendorID",           col("VendorID").cast(IntegerType()))
    .withColumn("passenger_count",    col("passenger_count").cast(IntegerType()))
    .withColumn("RatecodeID",         col("RatecodeID").cast(IntegerType()))
    .withColumn("PULocationID",       col("PULocationID").cast(IntegerType()))
    .withColumn("DOLocationID",       col("DOLocationID").cast(IntegerType()))
    .withColumn("payment_type",       col("payment_type").cast(IntegerType()))
    # Normalise flag
    .withColumn("store_and_fwd_flag", upper(trim(col("store_and_fwd_flag"))))
    # Monetary - round to 2 decimal places
    .withColumn("fare_amount",        spark_round(col("fare_amount").cast(DoubleType()), 2))
    .withColumn("tip_amount",         spark_round(col("tip_amount").cast(DoubleType()), 2))
    .withColumn("tolls_amount",       spark_round(col("tolls_amount").cast(DoubleType()), 2))
    .withColumn("total_amount",       spark_round(col("total_amount").cast(DoubleType()), 2))
    .withColumn("congestion_surcharge",spark_round(col("congestion_surcharge").cast(DoubleType()), 2))
    .withColumn("cbd_congestion_fee", spark_round(col("cbd_congestion_fee").cast(DoubleType()), 2))
    .withColumn("airport_fee",        spark_round(col("airport_fee").cast(DoubleType()), 2))
)

print("Cast schema:")
df_cast.printSchema()

Cast schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



### 3.2 - Domain Filtering

In [9]:
df_filtered = (
    df_cast
    .filter(col("fare_amount") > 0)
    .filter(col("trip_distance") >= 0)
    .filter(col("passenger_count").isNull() | col("passenger_count").between(1, 6))
    .filter(col("VendorID").isin(1, 2, 6, 7))
    .filter(col("RatecodeID").isin(1, 2, 3, 4, 5, 6, 99))
    .filter(col("payment_type").isin(0, 1, 2, 3, 4, 5, 6))
    .filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
    .filter(spark_year(col("tpep_pickup_datetime")) == 2026)
    .filter(col("store_and_fwd_flag").isin("Y", "N"))
)

filtered_rows = df_filtered.count()
removed       = bronze_rows - filtered_rows
print(f"Bronze rows   : {bronze_rows:,}")
print(f"After filter  : {filtered_rows:,}")
print(f"Removed       : {removed:,}  ({removed/bronze_rows*100:.2f}%)")

Bronze rows   : 3,724,889
After filter  : 2,536,191
Removed       : 1,188,698  (31.91%)


### 3.3 - Deduplication on Natural Key

In [10]:
natural_key = cfg["schema"]["natural_key"]  # [VendorID, tpep_pickup_datetime, PULocationID]

window_dedup = Window.partitionBy(*natural_key).orderBy(col("tpep_dropoff_datetime").desc())

df_deduped = (
    df_filtered
    .withColumn("_rn", row_number().over(window_dedup))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

dedup_rows = df_deduped.count()
dupes      = filtered_rows - dedup_rows
print(f"After filter  : {filtered_rows:,}")
print(f"After dedup   : {dedup_rows:,}")
print(f"Duplicates    : {dupes:,}  ({dupes/max(filtered_rows,1)*100:.3f}%)")

After filter  : 2,536,191
After dedup   : 2,498,904
Duplicates    : 37,287  (1.470%)


### 3.4 - Derived Columns

In [11]:
df_silver = (
    df_deduped
    .withColumn("pickup_date",      to_date(col("tpep_pickup_datetime")))
    .withColumn("pickup_month",     date_format(col("tpep_pickup_datetime"), "yyyy-MM"))
    .withColumn("trip_duration_min",
        spark_round(
            (uts(col("tpep_dropoff_datetime")) - uts(col("tpep_pickup_datetime"))) / 60.0, 2
        )
    )
    .withColumn("avg_speed_mph",
        when(col("trip_duration_min") > 0,
             spark_round(col("trip_distance") / (col("trip_duration_min") / 60.0), 2)
        ).otherwise(lit(None))
    )
)

print("Silver schema:")
df_silver.printSchema()
df_silver.select(
    "VendorID", "tpep_pickup_datetime", "trip_distance",
    "fare_amount", "pickup_date", "trip_duration_min", "avg_speed_mph"
).show(5, truncate=False)

Silver schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup

### 3.5 - EXPLAIN Physical Plan (Proof)

In [12]:
buf = io.StringIO()
old_stdout = sys.stdout
sys.stdout = buf
df_silver.explain(extended=True)
sys.stdout = old_stdout
plan_str = buf.getvalue()

proof_path = PROOF_DIR / "plan_bronze_to_silver.txt"
with open(proof_path, "w") as f:
    f.write(" Bronze -> Silver : EXPLAIN extended=True \n")
    f.write(f"Generated : {datetime.datetime.now().isoformat()}\n\n")
    f.write(plan_str)

print(f"Plan saved : {proof_path}  ({proof_path.stat().st_size:,} bytes)")
print()
print(plan_str[:3000])


Plan saved : outputs\project\proof\plan_bronze_to_silver.txt  (34,110 bytes)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(avg_speed_mph, CASE WHEN '`>`('trip_duration_min, 0) THEN 'round('`/`('trip_distance, '`/`('trip_duration_min, 60.0)), 2) ELSE null END, None)]
+- Project [VendorID#265, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#266, trip_distance#4, RatecodeID#267, store_and_fwd_flag#271, PULocationID#268, DOLocationID#269, payment_type#270, fare_amount#272, extra#11, mta_tax#12, tip_amount#273, tolls_amount#274, improvement_surcharge#15, total_amount#275, congestion_surcharge#276, airport_fee#278, cbd_congestion_fee#277, pickup_date#329, pickup_month#330, round((cast((unix_timestamp(tpep_dropoff_datetime#2, yyyy-MM-dd HH:mm:ss, Some(Europe/Paris), true) - unix_timestamp(tpep_pickup_datetime#1, yyyy-MM-dd HH:mm:ss, Some(Europe/Paris), true)) as double) / cast(60.0 as double)), 2) AS trip_duration_min#331]
   +- Project [VendorID#265, tpep_pi

### 3.6 - Write Silver (partitioned by `pickup_date`)

In [13]:
t_write = time.time()

(
    df_silver
    .repartition(4, col("pickup_date"))
    .write
    .mode("overwrite")
    .partitionBy("pickup_date")
    .parquet(str(SILVER_DIR))
)

write_s = time.time() - t_write
n_files = len(list(SILVER_DIR.rglob("*.parquet")))
print(f"Silver written in {write_s:.1f}s")
print(f"Parquet files   : {n_files}")
print(f"Output dir      : {SILVER_DIR.resolve()}")

Silver written in 8.2s
Parquet files   : 32
Output dir      : C:\Users\Enzor\Downloads\outputs\project\silver


### 3.7 - Footprint Comparison & Metrics

In [14]:
#  Read back & verify 
df_silver_back = spark.read.parquet(str(SILVER_DIR))
silver_rows    = df_silver_back.count()

#  Size comparison 
silver_bytes = sum(f.stat().st_size for f in SILVER_DIR.rglob("*.parquet"))
silver_gb    = silver_bytes / 1e9
ratio        = silver_bytes / bronze_dest.stat().st_size

print(" Footprint: Bronze vs Silver ")
print(f"  Bronze : {bronze_dest.stat().st_size/1e6:.1f} MB")
print(f"  Silver : {silver_bytes/1e6:.1f} MB")
print(f"  Ratio  : {ratio:.3f}  (Silver / Bronze)")
print()

#  Per-month summary 
print(" Silver summary ")
df_silver_back.select(
    "pickup_month", "fare_amount", "trip_distance", "trip_duration_min", "avg_speed_mph"
).groupBy("pickup_month").agg(
    {"fare_amount":"avg", "trip_distance":"avg",
     "trip_duration_min":"avg", "avg_speed_mph":"avg"}
).orderBy("pickup_month").show()

print(f"Silver rows (read back) : {silver_rows:,}")

 Footprint: Bronze vs Silver 
  Bronze : 64.2 MB
  Silver : 65.6 MB
  Ratio  : 1.022  (Silver / Bronze)

 Silver summary 
+------------+------------------+----------------------+------------------+------------------+
|pickup_month|  avg(fare_amount)|avg(trip_duration_min)|avg(avg_speed_mph)|avg(trip_distance)|
+------------+------------------+----------------------+------------------+------------------+
|     2026-01|19.704624293139467|    17.333412381352872|11.456228048867777| 3.468846153692295|
|     2026-02|              15.6|                 18.12|              5.89|              1.78|
+------------+------------------+----------------------+------------------+------------------+

Silver rows (read back) : 2,498,904


In [15]:
#  Record Silver metrics 
metrics_silver = {
    "timestamp"  : datetime.datetime.now().isoformat(),
    "stage"      : "silver_etl",
    "rows"       : silver_rows,
    "size_gb"    : round(silver_gb, 4),
    "partitions" : 4,
    "note"       : (f"bronze={bronze_rows} removed={removed} dupes={dupes} "
                    f"ratio={ratio:.3f}")
}
with open(METRICS_FILE, "a", newline="") as mf:
    w = csv.DictWriter(mf, fieldnames=metrics_silver.keys())
    w.writerow(metrics_silver)

print("Silver metrics written ->", METRICS_FILE)
for k, v in metrics_silver.items():
    print(f"  {k:12s}: {v}")

Silver metrics written -> outputs\project\metrics\project_metrics_log.csv
  timestamp   : 2026-06-07T20:17:00.445696
  stage       : silver_etl
  rows        : 2498904
  size_gb     : 0.0656
  partitions  : 4
  note        : bronze=3724889 removed=1188698 dupes=37287 ratio=1.022


### Silver Layer - Summary 

| Item | Value |
|------|-------|
| Input (Bronze) | raw rows from Jan 2026 |
| Domain filter | invalid fares, bad timestamps, out-of-range values removed |
| Deduplication | on `(VendorID, tpep_pickup_datetime, PULocationID)` |
| Derived columns | `pickup_date`, `pickup_month`, `trip_duration_min`, `avg_speed_mph` |
| Partition strategy | `partitionBy("pickup_date")`, 4 Spark partitions |
| Physical plan proof | `proof/plan_bronze_to_silver.txt` |
| **Next** | **Gold Layer ->** (cell block 4) |


---
## 4. Gold Layer - Analytical Tables <a id='4'></a>

The Gold layer exposes **query-optimised aggregated tables** built from Silver.  
We produce four analytical tables, then compact and partition by `pickup_month`.

| Table | Granularity | Business use |
|-------|-------------|--------------|
| `gold_hourly_stats` | hour-of-day × pickup_month | Demand & revenue by hour |
| `gold_zone_stats` | PULocationID × pickup_month | Top pick-up zones |
| `gold_vendor_stats` | VendorID × pickup_month | Vendor performance |
| `gold_payment_stats` | payment_type × pickup_month | Payment mix & tip rate |


### 4.1 - Load Silver

In [16]:
df_s = spark.read.parquet(str(SILVER_DIR))
df_s.cache()
silver_count = df_s.count()
print(f"Silver rows loaded : {silver_count:,}")

Silver rows loaded : 2,498,904


### 4.2 - `gold_hourly_stats`

In [17]:
gold_hourly = (
    df_s
    .withColumn("hour_of_day", hour(col("tpep_pickup_datetime")))
    .groupBy("pickup_month", "hour_of_day")
    .agg(
        count("*").alias("trip_count"),
        spark_round(avg("fare_amount"),        2).alias("avg_fare"),
        spark_round(avg("trip_distance"),      2).alias("avg_distance_mi"),
        spark_round(avg("trip_duration_min"),  2).alias("avg_duration_min"),
        spark_round(avg("total_amount"),       2).alias("avg_total"),
        spark_round(fsum("total_amount"),      2).alias("total_revenue"),
        spark_round(avg("passenger_count"),    2).alias("avg_pax"),
    )
    .orderBy("pickup_month", "hour_of_day")
)

gold_hourly.show(5)
print(f"Hourly rows : {gold_hourly.count():,}")

+------------+-----------+----------+--------+---------------+----------------+---------+-------------+-------+
|pickup_month|hour_of_day|trip_count|avg_fare|avg_distance_mi|avg_duration_min|avg_total|total_revenue|avg_pax|
+------------+-----------+----------+--------+---------------+----------------+---------+-------------+-------+
|     2026-01|          0|     58494|   20.88|           4.01|           14.55|    30.98|   1811967.97|   1.35|
|     2026-01|          1|     38035|   18.37|           3.32|           12.71|    27.66|   1052233.76|   1.35|
|     2026-01|          2|     26111|    17.3|            3.0|           12.24|    26.24|    685264.82|   1.36|
|     2026-01|          3|     17900|   18.51|           3.33|           12.21|    27.51|    492506.84|   1.35|
|     2026-01|          4|     12758|   23.68|           4.72|           16.55|    32.99|     420894.3|    1.3|
+------------+-----------+----------+--------+---------------+----------------+---------+-------------+-

### 4.3 - `gold_zone_stats`

In [18]:
gold_zone = (
    df_s
    .groupBy("pickup_month", "PULocationID")
    .agg(
        count("*").alias("trip_count"),
        spark_round(avg("fare_amount"),       2).alias("avg_fare"),
        spark_round(avg("trip_distance"),     2).alias("avg_distance_mi"),
        spark_round(avg("trip_duration_min"), 2).alias("avg_duration_min"),
        spark_round(fsum("total_amount"),     2).alias("total_revenue"),
        spark_round(avg("tip_amount"),        2).alias("avg_tip"),
    )
    .orderBy("trip_count", ascending=False)
)

gold_zone.show(10)
print(f"Zone rows : {gold_zone.count():,}")

+------------+------------+----------+--------+---------------+----------------+-------------+-------+
|pickup_month|PULocationID|trip_count|avg_fare|avg_distance_mi|avg_duration_min|total_revenue|avg_tip|
+------------+------------+----------+--------+---------------+----------------+-------------+-------+
|     2026-01|         132|    138870|    61.7|          14.94|           38.36|1.102980847E7|   9.02|
|     2026-01|         237|    131769|   12.65|           1.67|           11.87|   2720647.51|   2.81|
|     2026-01|         236|    119748|    13.2|            1.8|           12.37|   2523936.85|   2.89|
|     2026-01|         161|    115251|   15.97|            2.2|           15.23|   2924216.68|    3.4|
|     2026-01|         186|     92960|   16.18|           2.18|           15.75|   2337056.86|   3.33|
|     2026-01|         162|     89521|   15.43|           2.16|           14.57|   2209723.72|   3.33|
|     2026-01|         142|     85637|   13.83|           2.03|          

### 4.4 - `gold_vendor_stats`

In [19]:
gold_vendor = (
    df_s
    .groupBy("pickup_month", "VendorID")
    .agg(
        count("*").alias("trip_count"),
        spark_round(avg("fare_amount"),       2).alias("avg_fare"),
        spark_round(avg("trip_distance"),     2).alias("avg_distance_mi"),
        spark_round(avg("trip_duration_min"), 2).alias("avg_duration_min"),
        spark_round(fsum("total_amount"),     2).alias("total_revenue"),
        spark_round(
            count(col("store_and_fwd_flag") == "Y") / count("*") * 100, 2
        ).alias("store_fwd_pct"),
    )
    .orderBy("VendorID")
)

gold_vendor.show()
print(f"Vendor rows : {gold_vendor.count():,}")

+------------+--------+----------+--------+---------------+----------------+-------------+-------------+
|pickup_month|VendorID|trip_count|avg_fare|avg_distance_mi|avg_duration_min|total_revenue|store_fwd_pct|
+------------+--------+----------+--------+---------------+----------------+-------------+-------------+
|     2026-01|       1|    554996|   20.44|           3.99|           22.87|1.573065713E7|        100.0|
|     2026-01|       2|   1943907|    19.5|           3.32|           15.75|5.718751137E7|        100.0|
|     2026-02|       2|         1|    15.6|           1.78|           18.12|        26.69|        100.0|
+------------+--------+----------+--------+---------------+----------------+-------------+-------------+

Vendor rows : 3


### 4.5 - `gold_payment_stats`

In [20]:
gold_payment = (
    df_s
    .groupBy("pickup_month", "payment_type")
    .agg(
        count("*").alias("trip_count"),
        spark_round(avg("tip_amount"),   2).alias("avg_tip"),
        spark_round(avg("fare_amount"),  2).alias("avg_fare"),
        spark_round(fsum("tip_amount"),  2).alias("total_tips"),
        spark_round(fsum("total_amount"),2).alias("total_revenue"),
    )
    .orderBy("payment_type")
)

gold_payment.show()
print(f"Payment rows : {gold_payment.count():,}")

+------------+------------+----------+-------+--------+----------+-------------+
|pickup_month|payment_type|trip_count|avg_tip|avg_fare|total_tips|total_revenue|
+------------+------------+----------+-------+--------+----------+-------------+
|     2026-01|           1|   2166131|   4.14|   19.68| 8976493.4|6.442647128E7|
|     2026-02|           1|         1|   5.34|    15.6|      5.34|        26.69|
|     2026-01|           2|    292370|    0.0|   19.58|    120.29|   7373365.07|
|     2026-01|           3|     11117|    0.0|   17.93|       2.6|    260029.11|
|     2026-01|           4|     29285|    0.0|   23.22|     90.12|    858303.04|
+------------+------------+----------+-------+--------+----------+-------------+

Payment rows : 5


### 4.6 - EXPLAIN Physical Plan Silver -> Gold (Proof)

In [21]:
buf = io.StringIO()
sys.stdout = buf
gold_zone.explain(extended=True)
sys.stdout = sys.__stdout__
plan_gold = buf.getvalue()

proof_gold_path = PROOF_DIR / "plan_silver_to_gold.txt"
with open(proof_gold_path, "w") as f:
    f.write(" Silver -> Gold (zone_stats) : EXPLAIN extended=True \n")
    f.write(f"Generated : {datetime.datetime.now().isoformat()}\n\n")
    f.write(plan_gold)

print(f"Plan saved : {proof_gold_path}  ({proof_gold_path.stat().st_size:,} bytes)")
print()
print(plan_gold[:3000])

### 4.7 - Write Gold Tables (partitioned by `pickup_month`, compacted)

In [22]:
t_write = time.time()

gold_tables = {
    "gold_hourly_stats"  : gold_hourly,
    "gold_zone_stats"    : gold_zone,
    "gold_vendor_stats"  : gold_vendor,
    "gold_payment_stats" : gold_payment,
}

for name, df_g in gold_tables.items():
    out_path = GOLD_DIR / name
    (
        df_g
        .coalesce(1)                    # compact - small aggregated tables
        .write
        .mode("overwrite")
        .partitionBy("pickup_month")
        .parquet(str(out_path))
    )
    n_files = len(list(out_path.rglob("*.parquet")))
    size_kb = sum(f.stat().st_size for f in out_path.rglob("*.parquet")) / 1e3
    print(f"  {name:25s}  {n_files} file(s)  {size_kb:.1f} KB")

write_s = time.time() - t_write
print(f"\nGold written in {write_s:.1f}s  -> {GOLD_DIR.resolve()}")

### 4.8 - Footprint: Silver -> Gold

In [23]:
silver_bytes = sum(f.stat().st_size for f in SILVER_DIR.rglob("*.parquet"))
gold_bytes   = sum(f.stat().st_size for f in GOLD_DIR.rglob("*.parquet"))

print(" Footprint ")
print(f"  Silver : {silver_bytes/1e6:.2f} MB")
print(f"  Gold   : {gold_bytes/1e6:.2f} MB  ({gold_bytes/silver_bytes*100:.1f}% of Silver)")
print(f"  Bronze : {BRONZE_DIR.glob('*.parquet') and sum(f.stat().st_size for f in BRONZE_DIR.glob('*.parquet'))/1e6:.2f} MB")

slo = gold_bytes / bronze_dest.stat().st_size
print(f"\nSLO parquet_vs_csv_ratio ≤ {cfg['slos']['parquet_vs_csv_ratio']} -> ratio={slo:.3f}  {'' if slo <= cfg['slos']['parquet_vs_csv_ratio'] else 'Warning'}")

In [24]:
#  Record Gold metrics 
metrics_gold = {
    "timestamp"  : datetime.datetime.now().isoformat(),
    "stage"      : "gold_etl",
    "rows"       : sum(df_g.count() for df_g in gold_tables.values()),
    "size_gb"    : round(gold_bytes / 1e9, 4),
    "partitions" : 1,
    "note"       : "4 analytical tables: hourly, zone, vendor, payment"
}
with open(METRICS_FILE, "a", newline="") as mf:
    w = csv.DictWriter(mf, fieldnames=metrics_gold.keys())
    w.writerow(metrics_gold)

print("Gold metrics written ->", METRICS_FILE)
for k, v in metrics_gold.items():
    print(f"  {k:12s}: {v}")

### Gold Layer - Summary 

| Table | Granularity | Key insight |
|-------|-------------|-------------|
| `gold_hourly_stats` | hour × month | Peak hours, revenue curve |
| `gold_zone_stats` | zone × month | Top pick-up locations |
| `gold_vendor_stats` | vendor × month | Vendor market share & performance |
| `gold_payment_stats` | payment_type × month | Credit card vs cash tip rates |

- All tables compacted to **1 file** per partition (`coalesce(1)`)  
- Partitioned by `pickup_month` for efficient downstream queries  
- Physical plan saved -> `proof/plan_silver_to_gold.txt`  
- **Next ->** Section 5 : Streaming Ingestion  


---
## 5. Streaming Ingestion - Structured Streaming with Windowed Aggregation <a id='5'></a>

**Approach:** We simulate a real-time ingestion stream by feeding the Silver Parquet files  
through Spark Structured Streaming (`readStream` with `format("parquet")`).  

| Parameter | Value |
|-----------|-------|
| Source | Silver Parquet directory |
| Window | 1 hour, slide 30 min |
| Watermark | 10 minutes |
| Aggregation | trip count, avg fare, total revenue per window × PULocationID |
| Output mode | `append` (requires watermark) |
| Sink | Parquet -> `streaming/` |
| Trigger | `processingTime = "30 seconds"` |
| Evidence | `query.lastProgress` printed + saved to `proof/` |


### 5.1 - Streaming Setup

In [25]:
STREAMING_DIR    = Path(cfg["paths"]["streaming"])
CHECKPOINT_DIR   = STREAMING_DIR / "checkpoints"
STREAMING_OUTPUT = STREAMING_DIR / "output"

STREAMING_DIR.mkdir(parents=True,    exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True,   exist_ok=True)
STREAMING_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Streaming output    :", STREAMING_OUTPUT.resolve())
print("Checkpoint dir      :", CHECKPOINT_DIR.resolve())

### 5.2 - Explicit Schema (required for readStream)

In [26]:
# Structured Streaming requires an explicit schema - inferred from Silver
silver_schema = spark.read.parquet(str(SILVER_DIR)).schema
print("Schema used for streaming source:")
print(silver_schema)

### 5.3 - Build Streaming Query

In [27]:
#  Read stream from Silver Parquet files 
df_stream_raw = (
    spark.readStream
    .schema(silver_schema)
    .option("maxFilesPerTrigger", 5)      # process 5 partition files per micro-batch
    .parquet(str(SILVER_DIR))
)

# Cast TIMESTAMP_NTZ fields to TIMESTAMP for streaming (watermark requires TIMESTAMP)
df_stream = (
    df_stream_raw
    .withColumn("tpep_pickup_datetime", col("tpep_pickup_datetime").cast("timestamp"))
    .withColumn("tpep_dropoff_datetime", col("tpep_dropoff_datetime").cast("timestamp"))
)

#  Apply watermark + windowed aggregation 
windowed_agg = (
    df_stream
    .withWatermark("tpep_pickup_datetime",
                   cfg["pipeline_params"]["streaming"]["watermark_delay"])
    .groupBy(
        window(
            col("tpep_pickup_datetime"),
            cfg["pipeline_params"]["streaming"]["window_duration"],
            cfg["pipeline_params"]["streaming"]["slide_duration"],
        ),
        col("PULocationID"),
        col("pickup_month"),
    )
    .agg(
        count("*").alias("trip_count"),
        spark_round(avg("fare_amount"),  2).alias("avg_fare"),
        spark_round(avg("trip_distance"),2).alias("avg_distance_mi"),
        spark_round(fsum("total_amount"),2).alias("total_revenue"),
    )
)

print("Streaming query plan:")
windowed_agg.explain()

### 5.4 - Start Query, Await Completion & Capture `lastProgress`

In [28]:
t_stream = time.time()

query = (
    windowed_agg
    .writeStream
    .outputMode("append")
    .format("parquet")
    .option("path",            str(STREAMING_OUTPUT))
    .option("checkpointLocation", str(CHECKPOINT_DIR))
    .trigger(processingTime=cfg["pipeline_params"]["streaming"]["trigger_interval"])
    .start()
)

print("Streaming query started - waiting for all files to be processed...")

# Wait until query processes all available data then stop
query.awaitTermination(timeout=120)   # 2 min max on a single small file

elapsed = time.time() - t_stream
print(f"\nQuery finished in {elapsed:.1f}s")

### 5.5 - Capture `query.lastProgress` (Evidence)

In [29]:
last_progress = query.lastProgress

if last_progress:
    progress_str = json.dumps(last_progress, indent=2, default=str)
else:
    progress_str = "{}"
    print("Warning  No progress captured (query may have ended before first trigger)")

# Save to proof/
proof_stream_path = PROOF_DIR / "streaming_last_progress.json"
with open(proof_stream_path, "w") as f:
    f.write(progress_str)

print(f"lastProgress saved -> {proof_stream_path}")
print()
print(progress_str[:2000])

### 5.6 - Read Streaming Output & Validate

In [30]:
stream_files = list(STREAMING_OUTPUT.rglob("*.parquet"))

if stream_files:
    df_stream_out = spark.read.parquet(str(STREAMING_OUTPUT))
    stream_rows   = df_stream_out.count()
    stream_bytes  = sum(f.stat().st_size for f in stream_files)

    print(f"Streaming output rows  : {stream_rows:,}")
    print(f"Streaming output files : {len(stream_files)}")
    print(f"Streaming output size  : {stream_bytes/1e3:.1f} KB")
    print()
    df_stream_out.orderBy("window").show(10, truncate=False)
else:
    stream_rows  = 0
    stream_bytes = 0
    print("No output files yet - query may not have completed a full trigger window.")
    print("    Run the cell above again with a longer timeout if needed.")

### 5.7 - SLO Check & Metrics

In [31]:
slo_latency = elapsed <= cfg["slos"]["streaming_latency_s"]
print(f"SLO streaming latency ≤ {cfg['slos']['streaming_latency_s']}s -> "
      f"{' MET' if slo_latency else 'Warning  exceeded'} ({elapsed:.1f}s)")

#  Metrics 
metrics_stream = {
    "timestamp"  : datetime.datetime.now().isoformat(),
    "stage"      : "streaming_ingestion",
    "rows"       : stream_rows,
    "size_gb"    : round(stream_bytes / 1e9, 6),
    "partitions" : len(stream_files),
    "note"       : (f"window=1h slide=30min watermark=10min "
                    f"elapsed={elapsed:.1f}s slo={'MET' if slo_latency else 'EXCEEDED'}")
}
with open(METRICS_FILE, "a", newline="") as mf:
    w = csv.DictWriter(mf, fieldnames=metrics_stream.keys())
    w.writerow(metrics_stream)

print("Streaming metrics written ->", METRICS_FILE)
for k, v in metrics_stream.items():
    print(f"  {k:12s}: {v}")

### Streaming Layer - Summary 

| Item | Value |
|------|-------|
| Source | Silver Parquet (readStream, `maxFilesPerTrigger=5`) |
| Watermark | 10 minutes on `tpep_pickup_datetime` |
| Window | 1 hour, slide 30 min |
| Aggregation | trip_count, avg_fare, avg_distance_mi, total_revenue |
| Output mode | `append` (watermark-compliant) |
| Sink | Parquet -> `streaming/output/` |
| Evidence | `proof/streaming_last_progress.json` |
| **Next ->** | **Section 6 : Text Processing** |


---
## 6. Text Processing - Tokenisation, TF-IDF & Inverted Index <a id='6'></a>

**Corpus construction:** each taxi trip is described as a free-text document combining  
rate code label, payment type label, vendor name, zone IDs and numeric context.  
This gives a textual corpus on which we can build NLP features.

| Step | Description |
|------|-------------|
| 6.1 | Build text corpus from structured fields |
| 6.2 | Tokenise & normalise (lowercase, split) |
| 6.3 | Remove stop-words |
| 6.4 | Build TF-IDF features (`HashingTF` + `IDF`) |
| 6.5 | Build inverted index (term -> doc_ids) |
| 6.6 | Query latency benchmark (SLO ≤ 2s) |
| 6.7 | Footprint: Parquet vs CSV |
| 6.8 | Metrics |


### 6.1 - Build Text Corpus from Structured Fields

In [32]:
TEXT_DIR = Path(cfg["paths"]["text"])
TEXT_DIR.mkdir(parents=True, exist_ok=True)

#  Label mappings as Spark expressions 
vendor_label = (
    when(col("VendorID") == 1, lit("creative_mobile_technologies"))
    .when(col("VendorID") == 2, lit("curb_mobility"))
    .when(col("VendorID") == 6, lit("myle_technologies"))
    .when(col("VendorID") == 7, lit("helix"))
    .otherwise(lit("unknown_vendor"))
)

rate_label = (
    when(col("RatecodeID") == 1,  lit("standard_rate"))
    .when(col("RatecodeID") == 2,  lit("jfk_airport"))
    .when(col("RatecodeID") == 3,  lit("newark_airport"))
    .when(col("RatecodeID") == 4,  lit("nassau_westchester"))
    .when(col("RatecodeID") == 5,  lit("negotiated_fare"))
    .when(col("RatecodeID") == 6,  lit("group_ride"))
    .otherwise(lit("unknown_rate"))
)

payment_label = (
    when(col("payment_type") == 0, lit("flex_fare"))
    .when(col("payment_type") == 1, lit("credit_card"))
    .when(col("payment_type") == 2, lit("cash"))
    .when(col("payment_type") == 3, lit("no_charge"))
    .when(col("payment_type") == 4, lit("dispute"))
    .when(col("payment_type") == 5, lit("unknown_payment"))
    .when(col("payment_type") == 6, lit("voided_trip"))
    .otherwise(lit("unknown_payment"))
)

store_label = (
    when(col("store_and_fwd_flag") == "Y", lit("store_and_forward"))
    .otherwise(lit("direct_connection"))
)

#  Build doc_id + text column 
df_silver_txt = spark.read.parquet(str(SILVER_DIR))

df_corpus = (
    df_silver_txt
    .withColumn("doc_id", monotonically_increasing_id())
    .withColumn("text", concat_ws(" ",
        lit("vendor"),     vendor_label,
        lit("rate"),       rate_label,
        lit("payment"),    payment_label,
        lit("pickup_zone"),col("PULocationID").cast("string"),
        lit("dropoff_zone"),col("DOLocationID").cast("string"),
        store_label,
        lit("month"),      col("pickup_month"),
        lit("distance"),   col("trip_distance").cast("string"),
        lit("fare"),       col("fare_amount").cast("string"),
        lit("duration"),   col("trip_duration_min").cast("string"),
    ))
    .select("doc_id", "text", "pickup_month", "PULocationID")
)

df_corpus.show(3, truncate=80)
print(f"Corpus size : {df_corpus.count():,} documents")

### 6.2 - Tokenise & Normalise

In [33]:
tokenizer = RegexTokenizer(
    inputCol="text",
    outputCol="tokens_raw",
    pattern=r"\\W+",       # split on non-word chars
    minTokenLength=cfg["pipeline_params"]["text"]["min_token_length"]
)

df_tokenized = tokenizer.transform(df_corpus)
df_tokenized.select("doc_id", "tokens_raw").show(3, truncate=80)

### 6.3 - Stop-word Removal

In [34]:
# Default English stop-words + domain-specific additions
domain_stops = ["vendor", "rate", "payment", "zone", "month",
                "distance", "fare", "duration", "pickup", "dropoff"]

remover = StopWordsRemover(
    inputCol="tokens_raw",
    outputCol="tokens",
    stopWords=StopWordsRemover.loadDefaultStopWords("english") + domain_stops
)

df_tokens = remover.transform(df_tokenized).select(
    "doc_id", "tokens", "pickup_month", "PULocationID"
)

df_tokens.show(3, truncate=80)

### 6.4 - TF-IDF Features (`HashingTF` + `IDF`)

In [35]:
NUM_FEATURES = cfg["pipeline_params"]["text"]["max_features"]

# TF
hashing_tf = HashingTF(
    inputCol="tokens",
    outputCol="tf_features",
    numFeatures=NUM_FEATURES
)
df_tf = hashing_tf.transform(df_tokens)

# IDF - fit on corpus
idf       = IDF(inputCol="tf_features", outputCol="tfidf_features", minDocFreq=2)
idf_model = idf.fit(df_tf)
df_tfidf  = idf_model.transform(df_tf).select(
    "doc_id", "tokens", "tfidf_features", "pickup_month", "PULocationID"
)

print("TF-IDF pipeline complete")
df_tfidf.show(3, truncate=60)

In [36]:
t0 = time.time()
tfidf_path = TEXT_DIR / "tfidf_features"
(
    df_tfidf
    .select("doc_id", "tfidf_features", "pickup_month", "PULocationID")
    .repartition(4)
    .write.mode("overwrite")
    .parquet(str(tfidf_path))
)
tfidf_write_s = time.time() - t0
tfidf_bytes   = sum(f.stat().st_size for f in tfidf_path.rglob("*.parquet"))
print(f"TF-IDF written : {tfidf_bytes/1e6:.2f} MB  in {tfidf_write_s:.1f}s  -> {tfidf_path}")

### 6.5 - Inverted Index (term -> doc_ids)

In [37]:
# Explode tokens -> (doc_id, term) then group by term
df_inverted = (
    df_tokens
    .select("doc_id", explode("tokens").alias("term"))
    .groupBy("term")
    .agg(
        collect_list("doc_id").alias("doc_ids"),
    )
    .withColumn("doc_freq", size(col("doc_ids")))
    .orderBy("doc_freq", ascending=False)
)

print("Top 20 terms by document frequency:")
df_inverted.show(20, truncate=40)
print(f"Index vocabulary size : {df_inverted.count():,} terms")

In [38]:
inv_path = TEXT_DIR / "inverted_index"
(
    df_inverted
    .write.mode("overwrite")
    .parquet(str(inv_path))
)
inv_bytes = sum(f.stat().st_size for f in inv_path.rglob("*.parquet"))
print(f"Inverted index written : {inv_bytes/1e6:.3f} MB  -> {inv_path}")

### 6.6 - Query Latency Benchmark (SLO ≤ 2s)

In [39]:
df_inv_loaded = spark.read.parquet(str(inv_path))
df_inv_loaded.cache()
_ = df_inv_loaded.count()   # warm up cache

query_terms = ["credit_card", "jfk_airport", "newark_airport",
               "curb_mobility", "standard_rate"]

print(f"{'Term':<25s}  {'doc_freq':>10s}  {'latency_s':>10s}  {'SLO':>5s}")
print("-" * 60)

latencies = []
for term in query_terms:
    t0  = time.time()
    row = df_inv_loaded.filter(col("term") == term).select("doc_freq").collect()
    lat = time.time() - t0
    latencies.append(lat)
    freq = row[0]["doc_freq"] if row else 0
    slo  = "" if lat <= cfg["slos"]["text_query_latency_s"] else "❌"
    print(f"  {term:<23s}  {freq:>10,}  {lat:>10.3f}s  {slo}")

avg_lat = sum(latencies) / len(latencies)
print(f"\nAvg latency : {avg_lat:.3f}s   SLO ≤ {cfg['slos']['text_query_latency_s']}s -> "
      f"{' MET' if avg_lat <= cfg['slos']['text_query_latency_s'] else '❌ NOT MET'}")

### 6.7 - Footprint: Parquet vs CSV

In [40]:
# Write a CSV version of the inverted index for comparison
csv_path = TEXT_DIR / "inverted_index_csv"
(
    df_inv_loaded
    .select("term", "doc_freq")   # only lightweight cols - doc_ids list is huge
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(csv_path))
)

parquet_kb = inv_bytes / 1e3
csv_bytes  = sum(f.stat().st_size for f in csv_path.rglob("*.csv"))
csv_kb     = csv_bytes / 1e3
ratio      = inv_bytes / csv_bytes if csv_bytes else 0

print(" Inverted Index Footprint ")
print(f"  Parquet : {parquet_kb:>8.1f} KB")
print(f"  CSV     : {csv_kb:>8.1f} KB")
print(f"  Ratio   : {ratio:.3f}  (Parquet / CSV)")
print(f"  SLO ≤ {cfg['slos']['parquet_vs_csv_ratio']} -> {' MET' if ratio <= cfg['slos']['parquet_vs_csv_ratio'] else 'Warning  N/A'}")


### 6.8 - Metrics

In [41]:
metrics_text = {
    "timestamp"  : datetime.datetime.now().isoformat(),
    "stage"      : "text_processing",
    "rows"       : df_inv_loaded.count(),
    "size_gb"    : round((tfidf_bytes + inv_bytes) / 1e9, 6),
    "partitions" : 4,
    "note"       : (f"vocab_size={df_inv_loaded.count()} "
                    f"avg_query_latency={avg_lat:.3f}s "
                    f"parquet_csv_ratio={ratio:.3f} "
                    f"num_features={NUM_FEATURES}")
}
with open(METRICS_FILE, "a", newline="") as mf:
    w = csv.DictWriter(mf, fieldnames=metrics_text.keys())
    w.writerow(metrics_text)

print("Text metrics written ->", METRICS_FILE)
for k, v in metrics_text.items():
    print(f"  {k:12s}: {v}")

### Text Processing - Summary 

| Item | Value |
|------|-------|
| Corpus | 1 document per trip - labels + numeric context |
| Tokeniser | `RegexTokenizer` (split on `\W+`, min length 3) |
| Stop-words | English defaults + 10 domain-specific terms |
| TF-IDF | `HashingTF` (10 000 features) + `IDF` (minDocFreq=2) |
| Inverted index | term -> doc_ids list, sorted by doc_freq |
| Output | `text/tfidf_features/` + `text/inverted_index/` (Parquet) |
| Query latency | avg measured vs SLO ≤ 2s |
| Footprint proof | Parquet vs CSV comparison |
| **Next ->** | **Section 7 : KMeans Clustering** |


---
## 7. Iterative Workload - KMeans & BisectingKMeans Clustering <a id='7'></a>

**Goal:** cluster taxi trips by behaviour profile using numerical features.  
We sweep k ∈ {3, 5, 7, 10, 15} for both KMeans and BisectingKMeans,  
track Silhouette score per configuration, and compare shuffle cost  
before vs after repartitioning.

| Step | Description |
|------|-------------|
| 7.1 | Feature preparation (`VectorAssembler` + `StandardScaler`) |
| 7.2 | **Before** partitioning baseline - KMeans sweep |
| 7.3 | Repartition by `PULocationID` (locality-aware) |
| 7.4 | **After** partitioning - KMeans + BisectingKMeans sweep |
| 7.5 | Silhouette score comparison table |
| 7.6 | Best-k cluster analysis |
| 7.7 | EXPLAIN plans before/after |
| 7.8 | Metrics |


### 7.1 - Feature Preparation

In [42]:
GOLD_DIR_PATH = Path(cfg["paths"]["gold"])

# Load Silver - drop rows with any null in feature columns
feature_cols = [
    "fare_amount", "trip_distance", "trip_duration_min",
    "avg_speed_mph", "PULocationID", "DOLocationID",
    "tip_amount", "tolls_amount", "passenger_count"
]

df_feat = (
    spark.read.parquet(str(SILVER_DIR))
    .select(["doc_id_tmp"] + feature_cols
            if "doc_id_tmp" in spark.read.parquet(str(SILVER_DIR)).columns
            else feature_cols)
    .dropna(subset=feature_cols)
    .withColumn("passenger_count", col("passenger_count").cast("double"))
)

# Add a surrogate id
from pyspark.sql.functions import monotonically_increasing_id
df_feat = df_feat.withColumn("doc_id", monotonically_increasing_id())

print(f"Feature rows (no nulls) : {df_feat.count():,}")
df_feat.show(3)

In [43]:
#  Assemble + scale 
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler    = StandardScaler(inputCol="raw_features", outputCol="features",
                           withMean=True, withStd=True)

prep_pipeline = Pipeline(stages=[assembler, scaler])
prep_model    = prep_pipeline.fit(df_feat)
df_scaled     = prep_model.transform(df_feat).select("doc_id", "features")
df_scaled.cache()
_ = df_scaled.count()   # materialise cache

print("Feature vector sample:")
df_scaled.show(3, truncate=60)

### 7.2 - BEFORE Repartitioning: KMeans Sweep
Baseline with default Spark partitioning (inherited from Silver read).


In [44]:
K_RANGE  = cfg["pipeline_params"]["clustering"]["k_range"]
MAX_ITER = cfg["pipeline_params"]["clustering"]["max_iter"]
SEED     = cfg["pipeline_params"]["clustering"]["seed"]

evaluator = ClusteringEvaluator(
    featuresCol="features", metricName="silhouette", distanceMeasure="squaredEuclidean"
)

results_before = []

print(f"{'k':>4}  {'algo':<20}  {'silhouette':>12}  {'time_s':>8}")
print("-" * 52)

for k in K_RANGE:
    for AlgoClass, algo_name in [(KMeans, "KMeans"), (BisectingKMeans, "BisectingKMeans")]:
        t0  = time.time()
        mdl = AlgoClass(featuresCol="features", predictionCol="prediction",
                        k=k, maxIter=MAX_ITER, seed=SEED).fit(df_scaled)
        df_pred   = mdl.transform(df_scaled)
        silhouette = evaluator.evaluate(df_pred)
        elapsed    = time.time() - t0

        results_before.append({
            "phase": "before", "k": k, "algo": algo_name,
            "silhouette": round(silhouette, 4), "time_s": round(elapsed, 2)
        })
        slo = "" if silhouette >= cfg["slos"]["clustering_silhouette_min"] else "Warning"
        print(f"{k:>4}  {algo_name:<20}  {silhouette:>12.4f}  {elapsed:>8.2f}s  {slo}")

print()
print(f"SLO silhouette ≥ {cfg['slos']['clustering_silhouette_min']}")

### 7.3 - Repartition by `PULocationID` (Locality-Aware)
Grouping trips from the same zone in the same partition reduces cross-partition  
shuffle during distance computations.


In [45]:
# Join location back for repartitioning key
df_feat_loc = (
    spark.read.parquet(str(SILVER_DIR))
    .select(feature_cols + ["PULocationID"])
    .dropna(subset=feature_cols)
    .withColumn("passenger_count", col("passenger_count").cast("double"))
    .withColumn("doc_id", monotonically_increasing_id())
)

# Re-apply prep pipeline with repartition
# Join location back for repartitioning key
df_feat_loc = (
    spark.read.parquet(str(SILVER_DIR))
    .select(*feature_cols)   # avoid duplicating PULocationID
    .dropna(subset=feature_cols)
    .withColumn("passenger_count", col("passenger_count").cast("double"))
    .withColumn("doc_id", monotonically_increasing_id())
)

# Re-apply prep pipeline with repartition
df_scaled_repartitioned = (
    prep_model.transform(df_feat_loc)
    .select("doc_id", "features", "PULocationID")
    .repartition(20, col("PULocationID"))   # locality-aware
)
df_scaled_repartitioned.cache()
_ = df_scaled_repartitioned.count()

print(f"Repartitioned partitions : {df_scaled_repartitioned.rdd.getNumPartitions()}")
print("Repartition complete ")
df_scaled_repartitioned.cache()
_ = df_scaled_repartitioned.count()

print(f"Repartitioned partitions : {df_scaled_repartitioned.rdd.getNumPartitions()}")
print("Repartition complete ")

### 7.4 - AFTER Repartitioning: KMeans + BisectingKMeans Sweep

In [46]:
df_after = df_scaled_repartitioned.select("doc_id", "features")

results_after = []

print(f"{'k':>4}  {'algo':<20}  {'silhouette':>12}  {'time_s':>8}")
print("-" * 52)

for k in K_RANGE:
    for AlgoClass, algo_name in [(KMeans, "KMeans"), (BisectingKMeans, "BisectingKMeans")]:
        t0  = time.time()
        mdl = AlgoClass(featuresCol="features", predictionCol="prediction",
                        k=k, maxIter=MAX_ITER, seed=SEED).fit(df_after)
        df_pred    = mdl.transform(df_after)
        silhouette = evaluator.evaluate(df_pred)
        elapsed    = time.time() - t0

        results_after.append({
            "phase": "after", "k": k, "algo": algo_name,
            "silhouette": round(silhouette, 4), "time_s": round(elapsed, 2)
        })
        slo = "" if silhouette >= cfg["slos"]["clustering_silhouette_min"] else "Warning"
        print(f"{k:>4}  {algo_name:<20}  {silhouette:>12.4f}  {elapsed:>8.2f}s  {slo}")

### 7.5 - Before vs After Comparison Table

In [47]:
all_results = results_before + results_after
df_results  = pd.DataFrame(all_results)

# Pivot: silhouette before vs after per (k, algo)
pivot = df_results.pivot_table(
    index=["k", "algo"], columns="phase",
    values=["silhouette", "time_s"]
).round(4)
pivot.columns = ["_".join(c) for c in pivot.columns]
pivot = pivot.reset_index()
pivot["silhouette_delta"] = (pivot["silhouette_after"] - pivot["silhouette_before"]).round(4)
pivot["speedup"]          = (pivot["time_s_before"] / pivot["time_s_after"]).round(3)

print(" Before vs After Repartitioning ")
print(pivot.to_string(index=False))

# Best k overall
best = df_results[df_results["phase"] == "after"].sort_values("silhouette", ascending=False).iloc[0]
print(f"\nBest configuration (after repartition): k={best['k']}  algo={best['algo']}  silhouette={best['silhouette']}")

### 7.6 - Best-k Cluster Analysis

In [48]:
best_k    = int(best["k"])
best_algo = best["algo"]

AlgoClass = KMeans if best_algo == "KMeans" else BisectingKMeans
best_model = AlgoClass(featuresCol="features", predictionCol="cluster",
                       k=best_k, maxIter=MAX_ITER, seed=SEED).fit(df_after)

df_clustered = best_model.transform(df_after)

# Join back to Silver for interpretability
df_silver_ids = (
    spark.read.parquet(str(SILVER_DIR))
    .select(feature_cols)
    .dropna(subset=feature_cols)
    .withColumn("passenger_count", col("passenger_count").cast("double"))
    .withColumn("doc_id", monotonically_increasing_id())
)

df_analysis = df_clustered.join(df_silver_ids, on="doc_id", how="inner")

print(f" Cluster profile (k={best_k}, {best_algo}) ")
df_analysis.groupBy("cluster").agg(
    {"fare_amount": "avg", "trip_distance": "avg",
     "trip_duration_min": "avg", "avg_speed_mph": "avg",
     "cluster": "count"}
).withColumnRenamed("count(cluster)", "trip_count")  .orderBy("cluster")  .show(truncate=False)

### 7.7 - EXPLAIN Plans Before/After Repartitioning (Proof)

In [49]:
buf = io.StringIO()
sys.stdout = buf
df_scaled.explain(extended=True)
sys.stdout = sys.__stdout__
plan_before = buf.getvalue()

buf = io.StringIO()
sys.stdout = buf
df_scaled_repartitioned.explain(extended=True)
sys.stdout = sys.__stdout__
plan_after = buf.getvalue()

proof_clust_path = PROOF_DIR / "plan_clustering_before_after.txt"
with open(proof_clust_path, "w") as f:
    f.write(" Clustering: EXPLAIN BEFORE repartition \n")
    f.write(f"Generated : {datetime.datetime.now().isoformat()}\n\n")
    f.write(plan_before)
    f.write("\n\n Clustering: EXPLAIN AFTER repartition \n\n")
    f.write(plan_after)

print(f"Plans saved -> {proof_clust_path}  ({proof_clust_path.stat().st_size:,} bytes)")

# Save results CSV to proof/
df_results.to_csv(PROOF_DIR / "clustering_results.csv", index=False)
print(f"Results CSV saved -> {PROOF_DIR / 'clustering_results.csv'}")

### 7.8 - Write Cluster Assignments to Gold

In [50]:
cluster_out = GOLD_DIR / "gold_cluster_assignments"
(
    df_clustered.select("doc_id", "cluster")
    .write.mode("overwrite")
    .parquet(str(cluster_out))
)
clust_bytes = sum(f.stat().st_size for f in cluster_out.rglob("*.parquet"))
print(f"Cluster assignments written : {clust_bytes/1e3:.1f} KB -> {cluster_out}")

### 7.9 - Metrics

In [51]:
best_sil_after  = max(r["silhouette"] for r in results_after)
best_sil_before = max(r["silhouette"] for r in results_before)

metrics_clust = {
    "timestamp"  : datetime.datetime.now().isoformat(),
    "stage"      : "clustering",
    "rows"       : df_after.count(),
    "size_gb"    : round(clust_bytes / 1e9, 6),
    "partitions" : 20,
    "note"       : (f"best_k={best_k} algo={best_algo} "
                    f"sil_before={best_sil_before} sil_after={best_sil_after} "
                    f"slo={'MET' if best_sil_after >= cfg['slos']['clustering_silhouette_min'] else 'NOT_MET'}")
}
with open(METRICS_FILE, "a", newline="") as mf:
    w = csv.DictWriter(mf, fieldnames=metrics_clust.keys())
    w.writerow(metrics_clust)

print("Clustering metrics written ->", METRICS_FILE)
for k, v in metrics_clust.items():
    print(f"  {k:12s}: {v}")

### Clustering - Summary 

| Item | Value |
|------|-------|
| Features | fare, distance, duration, speed, PU/DO zones, tip, tolls, pax (9 dims) |
| Preprocessing | `VectorAssembler` + `StandardScaler` (mean=0, std=1) |
| Algorithms | `KMeans` + `BisectingKMeans` |
| k sweep | {3, 5, 7, 10, 15} |
| Evaluation | Silhouette (squaredEuclidean) |
| Repartition | `repartition(20, PULocationID)` - locality-aware |
| Evidence | `proof/clustering_results.csv`, `proof/plan_clustering_before_after.txt` |
| SLO silhouette ≥ 0.25 | checked per configuration |
| **Next ->** | **Section 8 : LLM Data Preparation** |


---
## 8. LLM Data Preparation - Curated Dataset for Fine-tuning / RAG <a id='8'></a>

**Goal:** produce a clean, structured text dataset ready for LLM fine-tuning or  
Retrieval-Augmented Generation (RAG). We do **not** run a LLM - we engineer  
the data pipeline that feeds one.

| Step | Description |
|------|-------------|
| 8.1 | Build structured text documents `(doc_id, text, metadata)` |
| 8.2 | Quality filter - minimum length |
| 8.3 | Quality filter - deduplication |
| 8.4 | Quality filter - language detection (ASCII proxy) |
| 8.5 | Export to Parquet -> `llm_ready/` |
| 8.6 | SLO check (≥ 80 % pass rate) |
| 8.7 | Data Card |
| 8.8 | Metrics |


### 8.1 - Build Structured Text Documents

In [52]:
LLM_DIR = Path(cfg["paths"]["llm_ready"])
LLM_DIR.mkdir(parents=True, exist_ok=True)

#  Human-readable label helpers (same as Section 6) 
def vendor_lbl(c):
    return (when(c == 1, lit("Creative Mobile Technologies"))
            .when(c == 2, lit("Curb Mobility"))
            .when(c == 6, lit("Myle Technologies"))
            .when(c == 7, lit("Helix"))
            .otherwise(lit("Unknown Vendor")))

def rate_lbl(c):
    return (when(c == 1, lit("Standard Rate"))
            .when(c == 2, lit("JFK Airport"))
            .when(c == 3, lit("Newark Airport"))
            .when(c == 4, lit("Nassau or Westchester"))
            .when(c == 5, lit("Negotiated Fare"))
            .when(c == 6, lit("Group Ride"))
            .otherwise(lit("Unknown Rate")))

def pay_lbl(c):
    return (when(c == 0, lit("Flex Fare"))
            .when(c == 1, lit("Credit Card"))
            .when(c == 2, lit("Cash"))
            .when(c == 3, lit("No Charge"))
            .when(c == 4, lit("Dispute"))
            .when(c == 5, lit("Unknown"))
            .when(c == 6, lit("Voided Trip"))
            .otherwise(lit("Unknown")))

df_raw = spark.read.parquet(str(SILVER_DIR))

df_docs = (
    df_raw
    .withColumn("doc_id", monotonically_increasing_id())
    #  Natural language trip description 
    .withColumn("text", concat_ws(" ",
        lit("A taxi trip operated by"), vendor_lbl(col("VendorID")),
        lit("was recorded in"), col("pickup_month"),
        lit(". The passenger picked up from zone"),
        col("PULocationID").cast("string"),
        lit("and dropped off at zone"),
        col("DOLocationID").cast("string"),
        lit(". The trip covered"),
        col("trip_distance").cast("string"),
        lit("miles in approximately"),
        col("trip_duration_min").cast("string"),
        lit("minutes at an average speed of"),
        col("avg_speed_mph").cast("string"),
        lit("mph. The fare was $"),
        col("fare_amount").cast("string"),
        lit("with a total charge of $"),
        col("total_amount").cast("string"),
        lit(". Payment was made by"),
        pay_lbl(col("payment_type")),
        lit(". Rate code:"), rate_lbl(col("RatecodeID")),
        lit(". Store and forward:"), col("store_and_fwd_flag"),
    ))
    #  Metadata fields 
    .select(
        col("doc_id"),
        col("text"),
        col("pickup_month").alias("metadata_month"),
        col("PULocationID").alias("metadata_pu_zone"),
        col("DOLocationID").alias("metadata_do_zone"),
        col("VendorID").alias("metadata_vendor"),
        col("payment_type").alias("metadata_payment"),
        col("fare_amount").alias("metadata_fare"),
        col("trip_distance").alias("metadata_distance"),
    )
)

total_docs = df_docs.count()
print(f"Total documents built : {total_docs:,}")
df_docs.show(2, truncate=120)

### 8.2 - Quality Filter: Minimum Text Length

In [53]:
MIN_LENGTH = cfg["pipeline_params"]["llm_prep"]["min_doc_length"]

df_len_filtered = df_docs.filter(length(col("text")) >= MIN_LENGTH)
after_len = df_len_filtered.count()

print(f"Before length filter : {total_docs:,}")
print(f"After  length filter : {after_len:,}  (min {MIN_LENGTH} chars)")
print(f"Removed              : {total_docs - after_len:,}  ({(total_docs-after_len)/total_docs*100:.2f}%)")

### 8.3 - Quality Filter: Deduplication on Text

In [54]:
# Deduplicate on the text column (identical trip descriptions)
df_dedup_llm = df_len_filtered.dropDuplicates(["text"])
after_dedup  = df_dedup_llm.count()

print(f"Before dedup : {after_len:,}")
print(f"After  dedup : {after_dedup:,}")
print(f"Removed      : {after_len - after_dedup:,}  ({(after_len-after_dedup)/after_len*100:.3f}%)")

### 8.4 - Quality Filter: Language Detection
We use an ASCII-ratio proxy: documents where > 95 % of characters are ASCII  
are considered English. Non-ASCII content would indicate encoding issues.


In [55]:
@udf(BooleanType())
def is_ascii_english(text: str) -> bool:
    if not text:
        return False
    ascii_chars = sum(1 for c in text if ord(c) < 128)
    return (ascii_chars / len(text)) >= 0.95

df_lang_filtered = df_dedup_llm.filter(is_ascii_english(col("text")))
after_lang = df_lang_filtered.count()

print(f"Before language filter : {after_dedup:,}")
print(f"After  language filter : {after_lang:,}")
print(f"Removed                : {after_dedup - after_lang:,}")

### 8.5 - Export to Parquet -> `llm_ready/`

In [56]:
t0 = time.time()

(
    df_lang_filtered
    .repartition(4)
    .write
    .mode("overwrite")
    .parquet(str(LLM_DIR))
)

llm_write_s = time.time() - t0
llm_files   = list(LLM_DIR.rglob("*.parquet"))
llm_bytes   = sum(f.stat().st_size for f in llm_files)

print(f"LLM-ready written in {llm_write_s:.1f}s")
print(f"Files   : {len(llm_files)}")
print(f"Size    : {llm_bytes/1e6:.2f} MB")
print(f"Output  : {LLM_DIR.resolve()}")

# Read back & verify schema
df_llm_back = spark.read.parquet(str(LLM_DIR))
df_llm_back.printSchema()
df_llm_back.show(2, truncate=120)

### 8.6 - SLO Check: Pass Rate ≥ 80 %

In [57]:
pass_rate = after_lang / total_docs
slo_met   = pass_rate >= cfg["slos"]["llm_quality_pass_rate"]

print(" LLM Data Quality Funnel ")
print(f"  Initial docs        : {total_docs:,}")
print(f"  After length filter : {after_len:,}   (removed {total_docs-after_len:,})")
print(f"  After dedup         : {after_dedup:,}  (removed {after_len-after_dedup:,})")
print(f"  After lang filter   : {after_lang:,}  (removed {after_dedup-after_lang:,})")
print(f"  Pass rate           : {pass_rate*100:.2f} %")
print()
print(f"SLO ≥ {cfg['slos']['llm_quality_pass_rate']*100:.0f}% pass rate -> "
      f"{' MET' if slo_met else '❌ NOT MET'}  ({pass_rate*100:.2f}%)")

### 8.7 - Data Card

In [59]:
data_card = f"""
# Data Card - LLM-Ready NYC Yellow Taxi Dataset

## Source
- **Origin:** NYC Taxi & Limousine Commission (TLC) - Yellow Taxi Trip Records
- **Period:** January 2026 (single month)
- **Pipeline stage:** Derived from Silver layer (cleaned, typed, deduplicated)

## Size
- **Documents (rows):** {after_lang:,}
- **Storage:** {llm_bytes/1e6:.2f} MB (Parquet / Snappy)
- **Pass rate through quality filters:** {pass_rate*100:.2f}%

## Schema
| Column | Type | Description |
|--------|------|-------------|
| doc_id | long | Unique document identifier |
| text | string | Natural language trip description (~100-150 chars) |
| metadata_month | string | Pickup month (yyyy-MM) |
| metadata_pu_zone | int | TLC Taxi Zone - pick-up |
| metadata_do_zone | int | TLC Taxi Zone - drop-off |
| metadata_vendor | int | Vendor ID (1,2,6,7) |
| metadata_payment | int | Payment type code |
| metadata_fare | double | Fare amount ($) |
| metadata_distance | double | Trip distance (miles) |

## Quality Filters Applied
1. **Minimum length:** text ≥ {MIN_LENGTH} characters
2. **Deduplication:** exact match on `text` column
3. **Language detection:** ASCII ratio ≥ 95% (English proxy)

## Intended Use
- **Fine-tuning:** instruction-following or domain-specific LLM on NYC taxi context
- **RAG:** embedding + retrieval over trip records for Q&A (e.g. "typical fare from zone 132?")
- **NOT suitable for:** PII-sensitive applications (no passenger names, no GPS coords)

## Limitations
- Synthetic-style descriptions: numeric fields formatted as text, no free-form driver notes
- Single month only: seasonal patterns not represented
- No GPS coordinates: zone-level granularity only
"""

print(data_card)

# Save to llm_ready/
data_card_path = LLM_DIR / "DATA_CARD.md"
with open(data_card_path, "w", encoding="utf-8") as f:
    f.write(data_card)
print(f"Data Card saved -> {data_card_path}")

### 8.8 - Metrics

In [60]:
metrics_llm = {
    "timestamp"  : datetime.datetime.now().isoformat(),
    "stage"      : "llm_data_prep",
    "rows"       : after_lang,
    "size_gb"    : round(llm_bytes / 1e9, 6),
    "partitions" : 4,
    "note"       : (f"total_docs={total_docs} pass_rate={pass_rate*100:.2f}% "
                    f"min_len={MIN_LENGTH} "
                    f"slo={'MET' if slo_met else 'NOT_MET'}")
}
with open(METRICS_FILE, "a", newline="") as mf:
    w = csv.DictWriter(mf, fieldnames=metrics_llm.keys())
    w.writerow(metrics_llm)

print("LLM metrics written ->", METRICS_FILE)
for k, v in metrics_llm.items():
    print(f"  {k:12s}: {v}")

### 8.9 - Full Pipeline Metrics Summary

In [61]:
df_metrics = pd.read_csv(METRICS_FILE)
print(" project_metrics_log.csv ")
print(df_metrics.to_string(index=False))

### LLM Data Preparation - Summary 

| Item | Value |
|------|-------|
| Format | `(doc_id, text, metadata_*)` - 9 columns |
| Text style | Natural language trip description per row |
| Filter 1 | Minimum length ≥ 50 chars |
| Filter 2 | Exact deduplication on `text` |
| Filter 3 | Language: ASCII ratio ≥ 95% |
| Pass rate | SLO ≥ 80% verified |
| Output | `llm_ready/` (Parquet/Snappy) + `DATA_CARD.md` |
| Intended use | Fine-tuning or RAG on NYC taxi domain |

---

## Pipeline Complete

All 5 components delivered in this single notebook:

| Component | Section | Evidence |
|-----------|---------|---------|
| **ETL Bronze->Silver->Gold** | §2 §3 §4 | `proof/plan_bronze_to_silver.txt`, `proof/plan_silver_to_gold.txt` |
| **Streaming** | §5 | `proof/streaming_last_progress.json` |
| **Text Processing** | §6 | `text/tfidf_features/`, `text/inverted_index/` |
| **Clustering (iterative)** | §7 | `proof/clustering_results.csv`, `proof/plan_clustering_before_after.txt` |
| **LLM Data Prep** | §8 | `llm_ready/`, `llm_ready/DATA_CARD.md` |


---
## 8. LLM Data Preparation - Curated Text Dataset for Fine-tuning / RAG <a id='8'></a>

**Goal:** produce a clean, structured text dataset ready for LLM fine-tuning or  
Retrieval-Augmented Generation (RAG). We do **not** run any LLM - we build the  
data pipeline that feeds one.

Each record becomes a natural-language **trip summary document** with:
- `doc_id` - unique identifier  
- `text`   - human-readable trip description  
- `metadata` - JSON blob (month, zone, vendor, payment, fare bucket)

| Step | Description |
|------|-------------|
| 8.1 | Build natural-language trip summaries |
| 8.2 | Quality filter 1 - minimum text length (≥ 50 chars) |
| 8.3 | Quality filter 2 - language detection (English only) |
| 8.4 | Quality filter 3 - deduplication on `text` |
| 8.5 | Assemble `(doc_id, text, metadata)` schema |
| 8.6 | Export to `llm_ready/` as Parquet |
| 8.7 | Data Card |
| 8.8 | Quality pass-rate check (SLO ≥ 80%) |
| 8.9 | Metrics |


### 8.1 - Build Natural-Language Trip Summaries

In [62]:
LLM_DIR = Path(cfg["paths"]["llm_ready"])
LLM_DIR.mkdir(parents=True, exist_ok=True)

df_src = spark.read.parquet(str(SILVER_DIR))

#  Human-readable labels 
vendor_lbl = (
    when(col("VendorID") == 1, lit("Creative Mobile Technologies"))
    .when(col("VendorID") == 2, lit("Curb Mobility"))
    .when(col("VendorID") == 6, lit("Myle Technologies"))
    .when(col("VendorID") == 7, lit("Helix"))
    .otherwise(lit("Unknown Vendor"))
)

rate_lbl = (
    when(col("RatecodeID") == 1,  lit("standard rate"))
    .when(col("RatecodeID") == 2,  lit("JFK airport rate"))
    .when(col("RatecodeID") == 3,  lit("Newark airport rate"))
    .when(col("RatecodeID") == 4,  lit("Nassau or Westchester rate"))
    .when(col("RatecodeID") == 5,  lit("negotiated fare"))
    .when(col("RatecodeID") == 6,  lit("group ride"))
    .otherwise(lit("unknown rate"))
)

pay_lbl = (
    when(col("payment_type") == 0, lit("flex fare"))
    .when(col("payment_type") == 1, lit("credit card"))
    .when(col("payment_type") == 2, lit("cash"))
    .when(col("payment_type") == 3, lit("no charge"))
    .when(col("payment_type") == 4, lit("dispute"))
    .when(col("payment_type") == 5, lit("unknown method"))
    .when(col("payment_type") == 6, lit("voided trip"))
    .otherwise(lit("unknown"))
)

fare_bucket = (
    when(col("fare_amount") < 10,  lit("short fare (under $10)"))
    .when(col("fare_amount") < 25,  lit("medium fare ($10-$25)"))
    .when(col("fare_amount") < 50,  lit("long fare ($25-$50)"))
    .otherwise(lit("premium fare (over $50)"))
)

#  Compose sentence 
df_text = (
    df_src
    .withColumn("doc_id", monotonically_increasing_id())
    .withColumn("text", concat_ws(" ",
        lit("A taxi trip operated by"), vendor_lbl,
        lit("picked up"),
        col("passenger_count").cast("string"),
        lit("passenger(s) from zone"),
        col("PULocationID").cast("string"),
        lit("and dropped them off at zone"),
        col("DOLocationID").cast("string"),
        lit("on"), col("pickup_date").cast("string"),
        lit("at"),
        format_string("%02d:00", col("pickup_hour")
            if "pickup_hour" in df_src.columns
            else lit(0)),
        lit(". The trip covered"),
        format_string("%.2f", col("trip_distance")),
        lit("miles in"),
        format_string("%.1f", col("trip_duration_min")),
        lit("minutes under the"),
        rate_lbl,
        lit(". The passenger paid by"),
        pay_lbl,
        lit("with a total charge of"),
        format_string("$%.2f", col("total_amount")),
        lit("("),
        fare_bucket,
        lit(")."),
    ))
)

df_text.select("doc_id", "text").show(3, truncate=120)

### 8.2 - Quality Filter 1 : Minimum Text Length (≥ 50 chars)

In [63]:
total_docs = df_text.count()
print(f"Total documents     : {total_docs:,}")

df_len = df_text.filter(length(col("text")) >= cfg["pipeline_params"]["llm_prep"]["min_doc_length"])
after_len = df_len.count()
print(f"After length filter : {after_len:,}  (removed {total_docs - after_len:,})")

### 8.3 - Quality Filter 2 : Language Detection
Our corpus is fully English by construction (all labels are hard-coded English strings).  
We verify by checking that each document contains at least one common English function word.


In [64]:
# Lightweight English check: must contain at least one of these marker words
english_markers = ["the", "and", "from", "zone", "trip", "fare", "paid"]
marker_pattern  = "|".join(english_markers)

df_lang = df_len.filter(
    flower(col("text")).rlike(marker_pattern)
)
after_lang = df_lang.count()
print(f"After language filter : {after_lang:,}  (removed {after_len - after_lang:,})")

### 8.4 - Quality Filter 3 : Deduplication on `text` (SHA-256 hash)

In [65]:
# Hash text -> keep first occurrence per hash
df_hashed = df_lang.withColumn("text_hash", sha2(col("text"), 256))

w_dedup = Window.partitionBy("text_hash").orderBy("doc_id")
df_dedup = (
    df_hashed
    .withColumn("_rn", row_number().over(w_dedup))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

after_dedup = df_dedup.count()
print(f"After dedup filter : {after_dedup:,}  (removed {after_lang - after_dedup:,} duplicates)")

### 8.5 - Assemble Final `(doc_id, text, metadata)` Schema

In [66]:
df_llm = (
    df_dedup
    .withColumn("metadata", to_json(struct(
        col("pickup_month"),
        col("PULocationID").alias("pickup_zone"),
        col("DOLocationID").alias("dropoff_zone"),
        col("VendorID").alias("vendor_id"),
        col("payment_type"),
        col("fare_amount"),
        col("trip_distance"),
        col("trip_duration_min"),
        col("text_hash"),
    )))
    .select("doc_id", "text", "metadata")
)

print("LLM-ready schema:")
df_llm.printSchema()
df_llm.show(3, truncate=100)
print(f"\nFinal LLM-ready documents : {df_llm.count():,}")

### 8.6 - Export to `llm_ready/` as Parquet

In [67]:
t0 = time.time()
(
    df_llm
    .repartition(4)
    .write.mode("overwrite")
    .parquet(str(LLM_DIR))
)
llm_write_s = time.time() - t0
llm_bytes   = sum(f.stat().st_size for f in LLM_DIR.rglob("*.parquet"))
llm_rows    = df_llm.count()

print(f"Written in  : {llm_write_s:.1f}s")
print(f"Output size : {llm_bytes/1e6:.2f} MB")
print(f"Documents   : {llm_rows:,}")
print(f"Output dir  : {LLM_DIR.resolve()}")

### 8.7 - Data Card

In [69]:
data_card = f"""
# Data Card - NYC Yellow Taxi LLM-Ready Dataset

## Source
- **Original dataset:** NYC TLC Yellow Taxi Trip Records - January 2026
- **Raw file:** {RAW_FILE.name} (~64 MB Parquet)
- **Pipeline:** Bronze -> Silver -> LLM Prep (DE2_Project_Notebook_EN.ipynb)

## Size
- **Documents:** {llm_rows:,}
- **Parquet size:** {llm_bytes/1e6:.2f} MB
- **Partitions:** 4

## Schema
| Field    | Type   | Description |
|----------|--------|-------------|
| doc_id   | long   | Unique document identifier (monotonically increasing) |
| text     | string | Natural-language trip summary (~120-160 chars per doc) |
| metadata | string | JSON blob: pickup_month, zones, vendor_id, payment_type, fare, distance, duration, text_hash |

## Quality Filters Applied
| Filter | Threshold | Removed |
|--------|-----------|---------|
| Minimum text length | ≥ 50 characters | {total_docs - after_len:,} docs |
| Language detection  | English marker words | {after_len - after_lang:,} docs |
| Deduplication       | SHA-256 hash on text | {after_lang - after_dedup:,} docs |
| **Total removed**   | | **{total_docs - llm_rows:,}** ({(total_docs - llm_rows)/total_docs*100:.2f}%) |
| **Pass rate**       | SLO ≥ 80% | **{llm_rows/total_docs*100:.2f}%** |

## Intended Use
- **LLM fine-tuning:** instruction-tuning on NYC taxi domain (fare estimation, trip Q&A)
- **RAG (Retrieval-Augmented Generation):** index trip summaries for semantic search
- **NOT intended for:** PII-sensitive applications (no passenger names/IDs present)

## Limitations
- Single month (January 2026) - seasonal bias possible
- Synthetic-style text (template-generated, not free-form human writing)
- No demographic or personal data included

## Licence
NYC TLC open data - public domain.
"""

data_card_path = LLM_DIR / "DATA_CARD.md"
with open(data_card_path, "w", encoding="utf-8") as f:
    f.write(data_card)

print(data_card)
print(f"\nData Card saved -> {data_card_path}")

### 8.8 - Quality Pass-Rate SLO Check

In [70]:
pass_rate = llm_rows / total_docs
slo_met   = pass_rate >= cfg["slos"]["llm_quality_pass_rate"]

print(f"Total input docs  : {total_docs:,}")
print(f"LLM-ready docs    : {llm_rows:,}")
print(f"Pass rate         : {pass_rate*100:.2f}%")
print(f"SLO ≥ {cfg['slos']['llm_quality_pass_rate']*100:.0f}%         -> {' MET' if slo_met else '❌ NOT MET'}")

### 8.9 - Metrics

In [71]:
metrics_llm = {
    "timestamp"  : datetime.datetime.now().isoformat(),
    "stage"      : "llm_data_prep",
    "rows"       : llm_rows,
    "size_gb"    : round(llm_bytes / 1e9, 6),
    "partitions" : 4,
    "note"       : (f"total_input={total_docs} pass_rate={pass_rate*100:.2f}% "
                    f"removed_len={total_docs-after_len} "
                    f"removed_lang={after_len-after_lang} "
                    f"removed_dedup={after_lang-after_dedup} "
                    f"slo={'MET' if slo_met else 'NOT_MET'}")
}
with open(METRICS_FILE, "a", newline="") as mf:
    w = csv.DictWriter(mf, fieldnames=metrics_llm.keys())
    w.writerow(metrics_llm)

print("LLM metrics written ->", METRICS_FILE)
for k, v in metrics_llm.items():
    print(f"  {k:12s}: {v}")

### 8.10 - Full Pipeline Metrics Summary

In [72]:
df_metrics = pd.read_csv(METRICS_FILE)
print(" project_metrics_log.csv ")
print(df_metrics.to_string(index=False))

### LLM Data Preparation - Summary 

| Item | Value |
|------|-------|
| Input | Silver - - all trips from January 2026 |
| Text format | Natural-language trip summary (template-generated) |
| Quality filters | Length ≥ 50 chars · English markers · SHA-256 dedup |
| Pass rate | ≥ SLO 80% checked |
| Output schema | `doc_id` (long), `text` (string), `metadata` (JSON string) |
| Output path | `llm_ready/` (Parquet, 4 partitions) |
| Data Card | `llm_ready/DATA_CARD.md` |
| Intended use | LLM fine-tuning or RAG indexing |

---

##  Pipeline Complete

All 8 components implemented in this single notebook:

| # | Component | Output |
|---|-----------|--------|
| 1 | Config & Spark | Session ready |
| 2 | **Bronze** | Raw immutable Parquet |
| 3 | **Silver** | Cleaned, typed, deduplicated |
| 4 | **Gold** | 4 analytical tables |
| 5 | **Streaming** | Windowed aggregation -> Parquet append |
| 6 | **Text** | TF-IDF + inverted index |
| 7 | **Clustering** | KMeans sweep, before/after repartition |
| 8 | **LLM Prep** | Curated text dataset + Data Card |


In [73]:
spark.stop()
print("Spark session stopped")